In [1]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn import functional as F
import numpy as np

import random
from typing import Tuple, List, Optional
import matplotlib.pyplot as plt
from datetime import datetime
import copy


import lightning as pl
from lightning.pytorch.loggers import WandbLogger
import wandb
import pprint

import monai
from monai.losses import DiceLoss, DiceCELoss, SoftclDiceLoss, DiceFocalLoss, NACLLoss
from kornia.losses import total_variation

from Models.model import MLP, initialize_siren_weights, SineLayer, ReLULayer
from Models.model_trainer import ModelTrainerModule



from Models.models import Siren, Finer

from Utils.utils import get_full_img, norm, get_device, dice_stack_helper, get_model, ClearCache
from Data.load_ixi import load_data
from Utils.defaults import default_config
from Utils.plotting_utils2 import plot_seg_results_paper, plot_final_results_paper, plot_hf_results_paper
from Utils.plotting_utils import loss_plot, plot_image_metrics, plot_4_images
from LFSynth.ContrastModulation import ContrastModulation

from Data.patchwise3D import RandomPointsDataset
from onnxruntime import InferenceSession
import onnx

In [2]:
POINTS_PER_SAMPLE = 96*96*4
lf_points_per_sample = 48*48*4

In [3]:
config = copy.deepcopy(default_config)
config["in_features"] = 3
config["l1"] = 100 #wandb.config.l1
config["l3"] = 8 #wandb.config.l3
config["l4"] = 0.075
config["l5"] = 0.09
config["is_new_contrast"] = False
hf_ground_truth, lf_gt, lf_gt_seg_dice, M = load_data(102, config) #uncomment
gt_image = torch.tensor(norm(hf_ground_truth)).unsqueeze(-1)
gt_image = gt_image.to(torch.float32)
lf_gt = torch.tensor(norm(lf_gt)).unsqueeze(-1)
lf_gt = lf_gt.to(torch.float32)
print("gt_image: ", gt_image.shape, "lf_gt: ", lf_gt.shape, "lf_gt_seg_dice: ", lf_gt_seg_dice.shape)
print('gt_image, lf_gt loaded')


dataset = RandomPointsDataset(gt_image, lf_gt, lf_gt_seg_dice, points_num=POINTS_PER_SAMPLE)
dataloader = DataLoader(dataset, batch_size=1, num_workers=0, pin_memory=False) # We set a batch_size of 1 since our dataloader is already returning a batch of points.
temp = next(iter(dataloader))

SNRs- WM:  63.633989918981285 GM:  50.15986626698883 CSF:  24.10768620369957 BG (std):  5.173624347484933
CNRs- WG:  13.474123651992457 WC:  39.52630371528171 GC:  26.05218006328926
known_m =  [0.5 0.6 0.2] c vector:  [ 1.7210752  26.99545772 25.27438252]
SNR matrix: [[ 63.63398992 -50.15986627   0.        ]
 [ 63.63398992   0.         -24.1076862 ]
 [  0.          50.15986627 -24.1076862 ]] Target Contrast:  [ 9.03 27.17 18.14]


delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.

A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


Solution :  [0.59062031 0.56925046 0.4319588 ]
Target Contrast:  [ 9.03 27.17 18.14] Achieved contrast:  [ 9.03       27.16999998 18.13999998]
gt_image:  torch.Size([182, 218, 182, 1]) lf_gt:  torch.Size([91, 109, 182, 1]) lf_gt_seg_dice:  torch.Size([1, 4, 91, 109, 182])
gt_image, lf_gt loaded
Device =  mps
torch.Size([4, 36864, 1])
torch.Size([4, 9216])


In [ ]:
plt.imshow(lf_gt[:,:,95,0], cmap='gray')
plt.colorbar()

In [ ]:
model_loc = "/Users/pi58/Library/CloudStorage/Box-Box/PhD/MPhil/Projects/Hulf_Synth/saved_models/model_best2.onnx"
onnx_model = onnx.load(model_loc)
onnx.checker.check_model(onnx_model)

onnx_model.graph.input[0].type.tensor_type.shape.dim[0].ClearField('dim_value')
onnx_model.graph.input[0].type.tensor_type.shape.dim[1].ClearField('dim_value')
# onnx_model.graph.input[0].type.tensor_type.shape.dim[2].ClearField('dim_value')
# onnx_model.graph.input[0].type.tensor_type.shape.dim[3].ClearField('dim_value')
sess = InferenceSession(onnx_model.SerializeToString())

In [ ]:
onnx_model.graph.input[0].type.tensor_type.shape

In [ ]:
dummy_input, _ , _ = next(iter(dataloader))
dummy_input = dummy_input.view(-1, dummy_input.shape[-1])
sess.run(None, {"onnx::Gemm_0" : dummy_input.cpu().numpy()})

In [ ]:
resolution = hf_ground_truth.shape
device = 'cpu' #get_device()
meshgrid = torch.meshgrid([torch.arange(0, i, device=device) for i in resolution], indexing='ij')
coords = torch.stack(meshgrid, dim=-1)
coords_norm = coords / torch.tensor(resolution, device=device) * 2 - 1
coords_norm_ = coords_norm.reshape(-1, coords.shape[-1])
predictions_, _, pred_seg_, _ = sess.run(None, {"onnx::Gemm_0" : coords_norm_.cpu().numpy()})
predictions = predictions_.reshape(resolution)
resolution_seg = list(resolution) + [pred_seg_.shape[-1]] #adding num_tissues to the resolution shape
pred_seg_ = pred_seg_.reshape(resolution_seg)
pred_seg = [pred_seg_[:,:,:,i].reshape(resolution) for i in range(pred_seg_.shape[-1])]
pred_seg = np.stack(pred_seg, axis = 0)

In [ ]:
plt.imshow((predictions[:,:,95]).T,cmap='gray')
plt.gca().invert_yaxis()
plt.colorbar()

In [ ]:
from Data.load_data_3d import get_gt_seg
(wm_gt_seg, gm_gt_seg, csf_gt_seg, bg_gt_seg) = get_gt_seg(1, config)
images_gt = [bg_gt_seg[:,:,95], wm_gt_seg[:,:,95], gm_gt_seg[:,:,95], csf_gt_seg[:,:,95]]
titles = ['BG', 'WM', 'GM', 'CSF']
plot_4_images_row(images, titles=titles, suptitle='Observed Segmentations')
plt.show()

In [ ]:
def plot_8_images_2rows(images, titles=None, figsize=(16, 8), cmap='gray', suptitle=None, row_labels=None, save_path=None, dpi=100):
    """
    Plot 8 images in 2 rows (4 images per row) using matplotlib.
    
    Parameters:
    -----------
    images : list or array-like
        List of 8 images to plot. Each image should be a numpy array.
        Can be 2D (grayscale) or 3D (RGB/color) arrays.
        Images 0-3 go in first row, images 4-7 go in second row.
    titles : list, optional
        List of 8 titles for each subplot. Default is None.
    figsize : tuple, optional
        Figure size as (width, height). Default is (16, 8).
    cmap : str, optional
        Colormap for grayscale images. Default is 'gray'.
    suptitle : str, optional
        Overall title for the entire figure. Default is None.
    row_labels : list, optional
        List of 2 labels for each row (appears on left side). Default is None.
    save_path : str, optional
        Path to save the figure. If None, figure is not saved.
    dpi : int, optional
        DPI for saved figure. Default is 100.
    
    Returns:
    --------
    fig, axes : matplotlib figure and axes objects (2x4 array)
    
    Example:
    --------
    # With 8 random images
    images = [np.random.rand(100, 100) for _ in range(8)]
    titles = [f'Image {i+1}' for i in range(8)]
    plot_8_images_2rows(images, titles=titles, suptitle='8 Sample Images')
    plt.show()
    """
    
    if len(images) != 8:
        raise ValueError("Exactly 8 images must be provided")
    
    # Create figure and subplots (2 rows, 4 columns)
    fig, axes = plt.subplots(2, 4, figsize=figsize)
    
    # Set overall title if provided
    if suptitle:
        fig.suptitle(suptitle, fontsize=16, fontweight='bold', y=0.98)
    
    # Plot images
    for i, img in enumerate(images):
        row_idx = i // 4  # Row index (0 or 1)
        col_idx = i % 4   # Column index (0, 1, 2, or 3)
        ax = axes[row_idx, col_idx]
        
        # Handle different image formats
        if len(img.shape) == 2:  # Grayscale
            im = ax.imshow(img, cmap=cmap)
        elif len(img.shape) == 3:  # Color
            if img.shape[2] == 1:  # Single channel
                im = ax.imshow(img.squeeze(), cmap=cmap)
            else:  # RGB/RGBA
                im = ax.imshow(img)
        else:
            raise ValueError(f"Image {i} has unsupported shape: {img.shape}")
        
        # Set title if provided
        if titles and i < len(titles):
            ax.set_title(titles[i], fontsize=12, pad=10)
        
        # Remove axes ticks
        ax.set_xticks([])
        ax.set_yticks([])
        
        # Add row labels on the leftmost column
        if col_idx == 0 and row_labels and row_idx < len(row_labels):
            ax.set_ylabel(row_labels[row_idx], fontsize=9, color='blue',
                         rotation=90, labelpad=10)
    
    # Adjust layout
    plt.tight_layout()
    
    # Save figure if path provided
    if save_path:
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight')
        print(f"Figure saved to: {save_path}")
    
    return fig, axes

In [ ]:
slice_num = 90
images_gt = [bg_gt_seg[:,:,slice_num], wm_gt_seg[:,:,slice_num], gm_gt_seg[:,:,slice_num], csf_gt_seg[:,:,slice_num]]
titles = ['BG', 'WM', 'GM', 'CSF']
images_pred = [pred_seg[0,:,:,slice_num], pred_seg[1,:,:,slice_num], pred_seg[2,:,:,slice_num], pred_seg[3,:,:,slice_num]]
# titles = ['BG', 'WM', 'GM', 'CSF']
plot_8_images_2rows(images_gt + images_pred, titles=titles, figsize=(12, 6), cmap='gray', suptitle='Segmentations ', row_labels=['observed' +' (slice: ' + str(slice_num) +')', 'predicted'+ '(slice: ' + str(slice_num) +')'], save_path=None, dpi=100)

In [ ]:
slice_num = 95
images_gt = [bg_gt_seg[:,:,slice_num], wm_gt_seg[:,:,slice_num], gm_gt_seg[:,:,slice_num], csf_gt_seg[:,:,slice_num]]
titles = ['BG', 'WM', 'GM', 'CSF']
images_pred = [pred_seg[0,:,:,slice_num], pred_seg[1,:,:,slice_num], pred_seg[2,:,:,slice_num], pred_seg[3,:,:,slice_num]]
# titles = ['BG', 'WM', 'GM', 'CSF']
plot_8_images_2rows(images_gt + images_pred, titles=titles, figsize=(12, 6), cmap='gray', suptitle='Segmentations ', row_labels=['observed' +' (slice: ' + str(slice_num) +')', 'predicted'+ '(slice: ' + str(slice_num) +')'], save_path=None, dpi=100)

In [ ]:
def plot_3_images_row(images, titles=None, figsize=(12, 4), cmap='gray',  suptitle=None, save_path=None, dpi=100, show_colorbar=False):
    """
    Plot 3 images in a single row using matplotlib.
    
    Parameters:
    -----------
    images : list or array-like
        List of 3 images to plot. Each image should be a numpy array.
        Can be 2D (grayscale) or 3D (RGB/color) arrays.
    titles : list, optional
        List of 3 titles for each subplot. Default is None.
    figsize : tuple, optional
        Figure size as (width, height). Default is (12, 4).
    cmap : str, optional
        Colormap for grayscale images. Default is 'gray'.
    suptitle : str, optional
        Overall title for the entire figure. Default is None.
    save_path : str, optional
        Path to save the figure. If None, figure is not saved.
    dpi : int, optional
        DPI for saved figure. Default is 100.
    show_colorbar : bool, optional
        Whether to show colorbar for grayscale images. Default is False.
    
    Returns:
    --------
    fig, axes : matplotlib figure and axes objects
    
    Example:
    --------
    # With random images
    images = [np.random.rand(100, 100) for _ in range(3)]
    titles = ['Original', 'Processed', 'Result']
    plot_3_images_row(images, titles=titles, suptitle='Image Processing Pipeline')
    plt.show()
    """
    
    if len(images) != 3:
        raise ValueError("Exactly 3 images must be provided")
    
    # Create figure and subplots
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    
    # Set overall title if provided
    if suptitle:
        fig.suptitle(suptitle, fontsize=16, fontweight='bold', y=1.02)
    
    # Plot each image
    for i, (ax, img) in enumerate(zip(axes, images)):
        # Handle different image formats
        if len(img.shape) == 2:  # Grayscale
            im = ax.imshow(img, cmap=cmap)
            if show_colorbar:
                plt.colorbar(im, ax=ax, shrink=0.8)
        elif len(img.shape) == 3:  # Color
            if img.shape[2] == 1:  # Single channel
                im = ax.imshow(img.squeeze(), cmap=cmap)
                if show_colorbar:
                    plt.colorbar(im, ax=ax, shrink=0.8)
            else:  # RGB/RGBA
                im = ax.imshow(img)
        else:
            raise ValueError(f"Image {i} has unsupported shape: {img.shape}")
        
        # Set title if provided
        if titles and i < len(titles):
            ax.set_title(titles[i], fontsize=12, pad=10, color = 'blue')
        
        # Remove axes ticks
        ax.set_xticks([])
        ax.set_yticks([])
    
    # Adjust layout
    plt.tight_layout()
    
    # Save figure if path provided
    if save_path:
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight')
        print(f"Figure saved to: {save_path}")
    
    return fig, axes

In [ ]:
slice_num = 95
final_img = (predictions * pred_seg[1]) + (predictions * pred_seg[2]) + (predictions * pred_seg[3]) #+ (predictions * pred_seg[0])
# plt.imshow(final_img[:,:,95], cmap='gray')

images = [norm(torch.from_numpy(hf_ground_truth))[:,:,slice_num] , predictions[:,:,slice_num], final_img[:,:,slice_num]]
# images2 = [norm(predictions * pred_seg[0])[:,:,slice_num] , norm((predictions * pred_seg[2]))[:,:,slice_num], norm((predictions * pred_seg[3]))[:,:,slice_num]]
titles = ['observed', 'predicted', 'predicted (weighted sum)']
plot_3_images_row(images, titles=titles, figsize=(12, 4), cmap='gray',  suptitle='High Field', save_path=None, dpi=100, show_colorbar=True)

In [ ]:
dice_score = monai.metrics.DiceMetric()
dice2 = monai.metrics.GeneralizedDiceScore()
iou_score = monai.metrics.MeanIoU()
psnr_value = monai.metrics.PSNRMetric(max_val = 1.0) #expects shape: BCHWD
ssim_value = monai.metrics.regression.SSIMMetric(spatial_dims=3, data_range=1.0) #expects shape: BCHWD

hf_gt_seg = torch.stack((bg_gt_seg, wm_gt_seg, gm_gt_seg, csf_gt_seg), axis = 0)


print("Dice (mean): ", dice_score(torch.from_numpy(pred_seg).unsqueeze(0), (hf_gt_seg).unsqueeze(0)).mean())
print("Dice (generalized): ", dice2(torch.from_numpy(pred_seg).unsqueeze(0), (hf_gt_seg).unsqueeze(0).to(torch.float32)).mean())
print("IoU (mean): ", iou_score(torch.from_numpy(pred_seg).unsqueeze(0), (hf_gt_seg).unsqueeze(0)).mean())
print('psnr: ', psnr_value( torch.from_numpy(predictions).unsqueeze(0).unsqueeze(0), norm(torch.from_numpy(hf_ground_truth)).unsqueeze(0).unsqueeze(0)))
# print('ssim: ', ssim_value( torch.from_numpy(predictions).unsqueeze(0).unsqueeze(0), norm(torch.from_numpy(hf_ground_truth)).unsqueeze(0).unsqueeze(0)))

In [ ]:
final_img = (predictions * pred_seg[1]) + (predictions * pred_seg[2]) + (predictions * pred_seg[3]) #+ (predictions * pred_seg[0])
print('psnr: ', psnr_value( torch.from_numpy(final_img).unsqueeze(0).unsqueeze(0), norm(torch.from_numpy(hf_ground_truth)).unsqueeze(0).unsqueeze(0)))
print('ssim: ', ssim_value( torch.from_numpy(final_img).unsqueeze(0).unsqueeze(0), norm(torch.from_numpy(hf_ground_truth)).unsqueeze(0).unsqueeze(0)))

Final Image (multiple views)

In [ ]:
# Create a figure with three subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
# data = gt_image[:,:,:,0]
# data = lf_gt_seg_dice[0,1,:,:,].detach().cpu()
data = final_img
dims = data.shape
# Coronal slice
coronal_slice_index = dims[1] // 2
coronal_slice = data[:, coronal_slice_index, :].T
axes[0].imshow(coronal_slice, cmap='gray')
axes[0].set_title("Coronal View")

axes[0].set_aspect('equal')
axes[0].invert_yaxis()

# Sagittal slice
sagittal_slice_index = dims[0] // 2
sagittal_slice = data[sagittal_slice_index, :, :].T
axes[1].imshow(sagittal_slice, cmap='gray')
axes[1].set_title("Sagittal View")

axes[1].set_aspect('equal')
axes[1].invert_yaxis()

# Axial slice
axial_slice_index = dims[2] // 2
axial_slice = data[:, :, axial_slice_index].T
axes[2].imshow(axial_slice, cmap='gray')
axes[2].set_title("Axial View")

axes[2].set_aspect('equal')
axes[2].invert_yaxis()

plt.tight_layout()

plt.show()

In [ ]:
import matplotlib.patches as mpatches

# Create a figure with three subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
# data = gt_image[:,:,:,0]
# data = lf_gt_seg_dice[0,1,:,:,].detach().cpu()
data1 = pred_seg[1]
data2 = pred_seg[2]
data3 = pred_seg[3]
dims = data1.shape
# Coronal slice
coronal_slice_index = dims[1] // 2
coronal_slice1 = data1[:, coronal_slice_index, :].T
coronal_slice2 = data2[:, coronal_slice_index, :].T
coronal_slice3 = data3[:, coronal_slice_index, :].T




axes[0].imshow(coronal_slice1, cmap='Greens')
axes[0].imshow(coronal_slice2, cmap='Reds', alpha = 0.9 * coronal_slice2)
axes[0].imshow(coronal_slice3, cmap='Blues', alpha = 0.9 * coronal_slice3)
axes[0].set_title("Coronal View")

axes[0].set_aspect('equal')
axes[0].invert_yaxis()

# Sagittal slice
sagittal_slice_index = dims[0] // 2
sagittal_slice1 = data1[sagittal_slice_index, :, :].T
sagittal_slice2 = data2[sagittal_slice_index, :, :].T
sagittal_slice3 = data3[sagittal_slice_index, :, :].T
axes[1].imshow(sagittal_slice1, cmap='Greens', alpha = 0.9 * sagittal_slice1)
axes[1].imshow(sagittal_slice2, cmap='Reds', alpha = 0.9 * sagittal_slice2)
axes[1].imshow(sagittal_slice3, cmap='Blues', alpha = 0.9 * sagittal_slice3)
axes[1].set_title("Sagittal View")

axes[1].set_aspect('equal')
axes[1].invert_yaxis()

# Axial slice
axial_slice_index = dims[2] // 2
axial_slice1 = data1[:, :, axial_slice_index].T
axial_slice2 = data2[:, :, axial_slice_index].T
axial_slice3 = data3[:, :, axial_slice_index].T

axes[2].imshow(axial_slice1, cmap='Greens', alpha = 0.9 * axial_slice1)
axes[2].imshow(axial_slice2, cmap='Reds', alpha = 0.9 * axial_slice2)
axes[2].imshow(axial_slice3, cmap='Blues', alpha = 0.9 * axial_slice3)
axes[2].set_title("Axial View")

axes[2].set_aspect('equal')
axes[2].invert_yaxis()


fig.suptitle("Segmentations (pve)", color='blue')
green_patch = mpatches.Patch(color='green', alpha=0.9, label='CSF')
red_patch = mpatches.Patch(color='red', alpha=0.9, label='GM')
blue_patch = mpatches.Patch(color='blue', alpha=0.9, label='CSF')

fig.legend(handles=[red_patch, blue_patch, green_patch])#, loc='lower right')
# fig.legend(handles=[red_patch, blue_patch], loc='center', fontsize=12)


plt.tight_layout()
plt.show()

In [ ]:
pred_seg.shap

In [ ]:
dataset = RandomPointsDataset(gt_image, lf_gt, lf_gt_seg_dice, points_num=POINTS_PER_SAMPLE)
dataloader = DataLoader(dataset, batch_size=1, num_workers=0, pin_memory=False) # We set a batch_size of 1 since our dataloader is already returning a batch of points.


In [ ]:
t = torch.randn(224, 224, 160)
print(t.shape)
b = (torch.tensor(t.shape)).view(-1)
b.shape
b